# Unix Variables

## Safety - Disable default variables

The default behavior is to assign the empty string to undeclared variables. This is a problem because it makes mis-spelled variables hard to detect.

In [1]:
echo $DOES_NOT_EXIST

In [2]:
set -u

In [3]:
echo $DOES_NOT_EXIST

bash: DOES_NOT_EXIST: unbound variable


: 1

## Assigning variables

In [4]:
FILENAME="temp.txt"

In [5]:
echo $FILENAME

temp.txt


In [6]:
echo "Some stuff" > $FILENAME

In [7]:
cat $FILENAME

Some stuff


## Common mistakes

You must not have spaces on either side of `=` in a variable assignment.

In [8]:
FILENAME = "temp.txt"

bash: FILENAME: command not found


: 127

Unix interprets this as: run a command called `FILENAME`

In [9]:
FILENAME ="temp.txt"

bash: FILENAME: command not found


: 127

Unix interprets this as: Assign space to the variable FILENAME then run a program called `temp.txt`

In [10]:
FILENAME= "temp.txt"

bash: ./temp.txt: Permission denied


: 126

## Using a variable

In [11]:
PREFIX="Gene"

In [12]:
echo $PREFIX

Gene


In [13]:
echo $PREFIX001

bash: PREFIX001: unbound variable


: 1

If you surround the variable name with curly braces, you can concatenate names.

In [14]:
echo ${PREFIX}001

Gene001


## Assigning command outputs to variables

To caputre the output of a command, use `$(command)`

In [15]:
FILES=$(ls)

In [16]:
echo $FILES

Unix01_File_And_Directory.ipynb Unix02_FileIO.ipynb Unix03_File_And_Text_Manipulation.ipynb Unix04_Varaibles.ipynb contents.txt temp.txt


In [17]:
grep -in "unix" $FILES | head -5

Unix01_File_And_Directory.ipynb:7:    "# Working with Unix"
Unix01_File_And_Directory.ipynb:69:      "Unix_Exercises.ipynb\n"
Unix01_File_And_Directory.ipynb:87:      "..\t\t\tUnix_Exercises.ipynb\n"
Unix01_File_And_Directory.ipynb:105:      "-rw-r--r--  1 cliburn  staff  15012 May 20 18:38 Unix_Exercises.ipynb\n"
Unix01_File_And_Directory.ipynb:154:      "Unix_Exercises.ipynb\n"


We can also use the anonumous caputre form.

In [18]:
grep -in "unix" $(ls) | head -5

Unix01_File_And_Directory.ipynb:7:    "# Working with Unix"
Unix01_File_And_Directory.ipynb:69:      "Unix_Exercises.ipynb\n"
Unix01_File_And_Directory.ipynb:87:      "..\t\t\tUnix_Exercises.ipynb\n"
Unix01_File_And_Directory.ipynb:105:      "-rw-r--r--  1 cliburn  staff  15012 May 20 18:38 Unix_Exercises.ipynb\n"
Unix01_File_And_Directory.ipynb:154:      "Unix_Exercises.ipynb\n"


You may sometimes see this old backticks form. It is equivalent although modern usage seems to favor the `$(command)` from.

In [19]:
grep -in "unix" `ls` | head -5

Unix01_File_And_Directory.ipynb:7:    "# Working with Unix"
Unix01_File_And_Directory.ipynb:69:      "Unix_Exercises.ipynb\n"
Unix01_File_And_Directory.ipynb:87:      "..\t\t\tUnix_Exercises.ipynb\n"
Unix01_File_And_Directory.ipynb:105:      "-rw-r--r--  1 cliburn  staff  15012 May 20 18:38 Unix_Exercises.ipynb\n"
Unix01_File_And_Directory.ipynb:154:      "Unix_Exercises.ipynb\n"


## Assigning results of  an arithmetic expression (integers only)

To do integer arithmetic, use `$(( expression ))`.

In [20]:
echo $((2 + 3))

5


This does not work for floating point numbers.

In [21]:
echo $((2.2 + 3.3))

bash: 2.2 + 3.3: syntax error: invalid arithmetic operator (error token is ".2 + 3.3")


: 1

You need to invoke the `bc` calculator program to deal with floating point numbers.

In [22]:
echo 2.2 + 3.3 | bc 

5.5


## Using variables in loops

In [23]:
for FILE in $(ls *txt)
do
    wc -c $FILE
done

     128 contents.txt
      11 temp.txt


In [24]:
for FIB in 1 1 2 3 5
do
    echo $FIB
done

1
1
2
3
5


In [25]:
for N in $(seq 1 10)
do
    if [[ $N -le 5 ]]
    then
        echo $N
    else
        echo $((3*N))
    fi
done

1
2
3
4
5
18
21
24
27
30


## Fibonacci series

Just for fun.

In [26]:
a=1
b=1
for i in $(seq 1 10)
do
    echo -n ${a}","
    tmp=$a
    a=$b
    b=$((tmp+b))
done

1,1,2,3,5,8,13,21,34,55,

## Single and double quotes

Variables are not evaluated within single quotes, but they are within double quotes.

In [60]:
FOO=42
echo '$FOO'

$FOO


In [63]:
FOO=42
echo "$FOO"

42


## Environment variables

You can see what variables are visible in the environment with `env`

In [27]:
env | head -5

LDFLAGS=-L/usr/local/opt/llvm/lib -Wl,-rpath,/usr/local/opt/llvm/lib:LDFLAGS=-L/usr/local/opt/openssl/lib
TERM_PROGRAM=Apple_Terminal
SPARK_HOME=/Users/cliburn/spark
SHELL=/bin/bash
TERM=xterm-256color


In [28]:
echo $HOME

/Users/cliburn


To make a variable visible in the general environment so that other programs can use it, you need to `export` it.

In [29]:
env | grep EXPORTED_VARIABLE

: 1

In [30]:
export EXPORTED_VARIABLE="Hello, Unix"

In [31]:
env | grep EXPORTED_VARIABLE

EXPORTED_VARIABLE=Hello, Unix


Now remove the environment variable.

In [32]:
unset EXPORTED_VARIABLE

In [33]:
env | grep EXPORTED_VARIABLE

: 1

## Brace expansion

Brace expansions create lists of strings. It can also generate ranges.

In [59]:
echo file.{c,cpp,py,ipynb,csv,txt}

file.c file.cpp file.py file.ipynb file.csv file.txt


In [48]:
echo {a..c}{1..3}.txt

a1.txt a2.txt a3.txt b1.txt b2.txt b3.txt c1.txt c2.txt c3.txt


In [51]:
for NUM in {1..3}; do
    echo mkdir EXPT-${NUM}
done

mkdir EXPT-1
mkdir EXPT-2
mkdir EXPT-3


## Shell scripts

A shell script is just a collection of shell commands that you are now familiar with put into a file that can be executed from the command line. There are a few steps to make a shell script.

1. The first line often contains instructions to use a shell
`#!/bin/bash`
2. The other lines contain standard variable declarations, shell commands, loops etc
3. Save the file with the extension (`.sh`)
4. Make the file executable by `chmod +x <FILENAME>.sh`

Now you can run the shell script as though it were a shell command.

### First shell script

Here we will show the mechanics of creating a shell script.

Use an editor to write `script01.sh`

```bash
#!/bin/bash

echo "Hello bash!"
```

In [5]:
ls *sh

Bash_Exercise_1_Solutions.sh	script01.sh
Bash_Exercise_2_Solutoins.sh


Change permission to make file executable.

In [6]:
chmod +x script01.sh

In [7]:
ls -l *sh

-rw-r--r--  1 cliburn  staff  1323 May 22 13:57 Bash_Exercise_1_Solutions.sh
-rw-r--r--  1 cliburn  staff  1126 May 22 13:57 Bash_Exercise_2_Solutoins.sh
-rwxr-xr-x  1 cliburn  staff    32 May 22 14:11 script01.sh


In [8]:
./script01.sh

Hello bash!


### Second shell script

Here we see how to pass arguments to a script in `script02.sh`

```bash
#!/bin/bash                                                                     

echo '$# gives $ of arguments     :' $#
echo '$@ gives arguments as array :' $@
echo '$* gives arguments as string:' $*

echo '$1, $2, $3 give firs, second, third arguments etc'
echo '$1:' $1
echo '$2:' $2
echo '$2:' $3

echo 'Evaluating "$@"'
for ARG in "$@"
do
    echo ${ARG}
done
echo 'Evaluating "$*"'
for ARG in $*
do
    echo ${ARG}
done


```

In [67]:
chmod +x script02.sh

In [68]:
./script02.sh a b "c d"

$# gives $ of arguments     : 3
$@ gives arguments as array : a b c d
$* gives arguments as string: a b c d
$1, $2, $3 give firs, second, third arguments etc
$1: a
$2: b
$2: c d
Evaluating "$@"
a
b
c d
Evaluating "$*"
a
b
c
d


## Exercises

1. Write a shell script that accepts an arbitrary number of filenames as arguments (possibly given by `ls`), and outputs the total number of words in those files.

```bash
#!/bin/bash

TOTAL=0
for FILE in "$@"
do
    N=$(wc -w < $FILE)
    TOTAL=$((TOTAL + N))
done
echo $TOTAL
```